In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from scipy.stats import norm

In [124]:



no_of_output_nodes = 1
no_of_assets = 5

# cor_mat = [[1.0, 1.0, 1.0, 1.0, 1.0],
#            [1.0, 1.0, 1.0, 1.0, 1.0],
#            [1.0, 1.0, 1.0, 1.0, 1.0],
#            [1.0, 1.0, 1.0, 1.0, 1.0],
#            [1.0, 1.0, 1.0, 1.0, 1.0]]

cor_mat = [[1.0, 0.79, 0.82, 0.91, 0.84],
           [0.79, 1.0, 0.73, 0.80, 0.76],
           [0.82, 0.73, 1.0, 0.77, 0.72],
           [0.91, 0.80, 0.77, 1.0, 0.90],
           [0.84, 0.76, 0.72, 0.90, 1.0]]
vol_list = np.array([0.518, 0.648, 0.623, 0.570, 0.530])
T = 1
K = 1
no_of_exercise_days = 1
r = 0.05
w = np.array([0.381, 0.065, 0.057, 0.270, 0.227])
w = w.reshape(-1, 1)
exercise_days = np.array([float(i / no_of_exercise_days) for i in range(1, no_of_exercise_days + 1)])
dt = T / no_of_exercise_days

In [3]:
def generate_covarinace_mat(corr_mat, volatilities, dt):
    return np.outer(volatilities, volatilities) * corr_mat * dt

In [141]:
def generate_multi_stock_price(S0, r, volatilities, cor_mat, dt, N):
    # Direct simulation using NumPy
    stock_prices = np.zeros((len(volatilities), N))
    stock_prices[:, 0] = np.log(S0) * np.ones(len(volatilities))
    mu = stock_prices[:, 0] + (r - q_vect -  0.5 * volatilities ** 2) * dt
    Sigma = generate_covarinace_mat(cor_mat, volatilities, dt)
    for i in range(1, N):
        stock_prices[:, i] = np.random.multivariate_normal(mu, Sigma)
        current_log_prices = stock_prices[:, i]
        mu = current_log_prices + (r - 0.5 * volatilities ** 2) * dt
    return np.exp(stock_prices)


In [5]:
stock_prices = generate_multi_stock_price(1, r, vol_list, cor_mat, dt, 2)

In [6]:
stock_prices[:, 1]

array([1.65688153, 3.29842017, 1.60896968, 1.78184751, 2.35448788])

In [7]:
def arithmatic_basket_option_price(S0, r, volatilities, cor_mat, dt, N, weights, K, M):
    option_prices = []
    for _ in range(M):
        stock_prices = generate_multi_stock_price(S0, r, volatilities, cor_mat, dt, N)
        price_T = stock_prices[:, 1]
        avg_ST = np.dot(weights.T, price_T)
        option_price = np.maximum(K - avg_ST, 0)
        option_prices.append(option_price)
    option_price = np.mean(option_prices) * np.exp(-r * dt)
    
    return option_price

In [8]:
arithmatic_basket_option_price(1, r, vol_list, cor_mat, dt, 2, w, K, 200000)

0.17594331401057053

In [9]:
def price_max_call_option(S0, r, volatilities, cor_mat, dt, N, K, M):
    option_prices = []
    for _ in range(M):
        stock_prices = generate_multi_stock_price(S0, r, volatilities, cor_mat, dt, N)
        price_T = stock_prices[:, 1]
        option_price = np.maximum(np.max(price_T) - K, 0)
        option_prices.append(option_price)
    option_price = np.mean(option_prices) * np.exp(-r * dt)
    
    return option_price

In [10]:
def multi_asset_NN(no_hidden_nodes):
    
    model = Sequential()
    model.add(Dense(no_hidden_nodes, activation='relu', kernel_initializer='random_uniform'))
    model.add(Dense(1, activation='linear', kernel_initializer='random_normal'))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    
    return model


In [187]:
def calculate_conti_value_Multi(w1, b1, w2, b2, stock_prices, volatilities, covariance_mat, r, dt):
    
    sample_size = stock_prices.shape[0]
    no_hidden_nodes = w1.shape[1]
    no_of_assets = stock_prices.shape[1]
    x = np.log(stock_prices) + np.tile(((r - q_vect - 0.5 * np.square(volatilities)) * dt).reshape(1, no_of_assets),
                                       (sample_size, 1))
    opt_val = np.zeros((sample_size, 1))

    for node in range(0, no_hidden_nodes):
        w_o = w1[:, node]
        w_o = w_o.reshape(no_of_assets, 1)
        mu = np.dot(x, w_o) + b1[node]
        var = np.dot(np.dot(w_o.T, covariance_mat), w_o)
        sd = var ** 0.5
        ft = mu * (1 - norm(0, sd).cdf(-mu))
        st = (sd / (2 * np.pi) ** 0.5) * np.exp(-0.5 * (mu / sd) ** 2)
        opt_val = opt_val + w2[node] * (ft + st)

    continuation_value = (opt_val + b2) * np.exp(-(r - 0.1) * dt)

    return continuation_value
    

In [12]:
N = 2
M = 50000
S0 = 1
K = 1
r = 0.05
T = 1
no_of_assets = 5
corr_mat = np.array([[1.0, 0.79, 0.82, 0.91, 0.84],
                     [0.79, 1.0, 0.73, 0.80, 0.76],
                     [0.82, 0.73, 1.0, 0.77, 0.72],
                     [0.91, 0.80, 0.77, 1.0, 0.90],
                     [0.84, 0.76, 0.72, 0.90, 1.0]])
vol_list = np.array([0.518, 0.648, 0.623, 0.570, 0.530])
dt = T / (N-1)


cor_mat = np.eye(no_of_assets)
cov_mat = generate_covarinace_mat(cor_mat, vol_list, dt)

no_hidden_nodes = 64
es = EarlyStopping(monitor='val_loss', mode='min', patience=10)

In [13]:
print(stock_prices.shape)

(50000, 5, 2)


In [130]:
def payoff_func_Multi(stock_prices, w, K, type='Basket'):
    if type == 'Basket':
        payoff = np.maximum(K - np.dot(stock_prices, w), 0)
    elif type == 'Max':
        payoff = np.maximum(np.max(stock_prices, axis=1).flatten() - K, 0)
    return payoff

In [71]:
def RLNN_pre_training(model, S0, r, vol_list, cor_mat, dt, N, M, K, type='Basket'):
    
    stock_prices = np.zeros((M, no_of_assets, N))
    for i in range(M):
        stock_prices[i] = generate_multi_stock_price(S0, r, vol_list, cor_mat, dt, N)
        
    stock_price_T = stock_prices[:, :, -1]
    X_train = np.log(stock_price_T)
    y_train = payoff_func_Multi(stock_price_T, w, K, type)
    
    model.fit(X_train, y_train, epochs=3000, validation_split=0.2, callbacks=[es], verbose=0)
    
    mse = model.evaluate(X_train, y_train)
    
    return model, mse

In [184]:
def RLNN_MultiAsset(M, no_of_assets, K, model, cor_mat, vol_list, no_of_exercise_days, S0, type_option='Max'):
   """M: Number of paths
      N: Number of time steps
      no_of_assets: Number of assets
      K: Strike price
      model: Neural network model"""
   # Creating zero n-d arrays for intrinsic value, continuation value and option value
   N = no_of_exercise_days + 1
   model, mse = RLNN_pre_training(model, S0, r, vol_list, cor_mat, dt, N, M, K, type_option)
   print(mse)
   model.optimizer.learning_rate.assign(5e-4)
   model, mse = RLNN_pre_training(model, S0, r, vol_list, cor_mat, dt, N, M, K, type_option)
   print(mse)
   cov_mat = generate_covarinace_mat(cor_mat, vol_list, dt)
   ## Main Training Starts Here
   stock_prices = np.zeros((M, no_of_assets, N))
   for i in range(M):
      stock_prices[i] = generate_multi_stock_price(S0, r, vol_list, cor_mat, dt, N)
      
   # Creating zero n-d arrays for intrinsic value, continuation value and option value
   no_of_paths = M
   continuation_value = np.zeros((no_of_paths, 1))
   stock_vec = stock_prices[:, :,  no_of_exercise_days]
   intrinsic_value = payoff_func_Multi(stock_vec, w, K, type_option)
   continuation_value = intrinsic_value
   
   for day in range(no_of_exercise_days - 1, -1, -1):
      
      stock_vec = stock_prices[:, :, day + 1]
      option_value = continuation_value
      
      X_train = np.log(stock_vec)
      Y_train = option_value
      Y_train = Y_train.flatten()

      model.fit(X_train, Y_train, epochs=2000, batch_size=int(M/10), verbose=0,
                              validation_split=0.2, callbacks=[es])

      print(f"mse at {day}", model.evaluate(X_train, Y_train))
      w1 = np.array(model.layers[0].get_weights()[0])
      w2 = np.array(model.layers[1].get_weights()[0])
      bias1 = np.array(model.layers[0].get_weights()[1])
      bias2 = np.array(model.layers[1].get_weights()[1])
      stock_vec = stock_prices[:, :,  day]
      
      continuation_value = calculate_conti_value_Multi(w1, bias1, w2, bias2, stock_vec, vol_list, cov_mat, r, dt)
      #print("day", day)
   #print(np.mean(continuation_value))
   
   return np.mean(continuation_value)
     
     

In [192]:
nn_model = multi_asset_NN(256)
no_of_paths = 30000
no_of_assets = 2
K = 1
T = 3
no_of_exercise_days = 9
dt = T / (no_of_exercise_days)
# cor_mat = [[1.0, 0.79, 0.82, 0.91, 0.84],
#            [0.79, 1.0, 0.73, 0.80, 0.76],
#            [0.82, 0.73, 1.0, 0.77, 0.72],
#            [0.91, 0.80, 0.77, 1.0, 0.90],
#            [0.84, 0.76, 0.72, 0.90, 1.0]]
#vol_list = np.array([0.518, 0.648, 0.623, 0.570, 0.530])
cor_mat = np.eye(no_of_assets)
vol_list = np.ones(no_of_assets) * 0.2
cov_mat = generate_covarinace_mat(cor_mat, vol_list, dt)
r = 0.05
q_vect = np.ones(no_of_assets) * 0.1
S0 = 1.1
RLNN_MultiAsset(no_of_paths, no_of_assets, K, nn_model, cor_mat, vol_list, no_of_exercise_days, S0, "Max")

938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 164us/step - loss: 3.2510e-05
3.491558527457528e-05
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 165us/step - loss: 2.2039e-05
2.339405364182312e-05
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 164us/step - loss: 3.5361e-05
mse at 8 2.45013834501151e-05
day 8
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 165us/step - loss: 4.0063e-05
mse at 7 4.05287355533801e-05
day 7
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 165us/step - loss: 2.4925e-05
mse at 6 2.5614506739657372e-05
day 6
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 166us/step - loss: 2.0077e-05
mse at 5 2.0082175979041494e-05
day 5
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 164us/step - loss: 1.5262e-05
mse at 4 1.496066488471115e-05
day 4
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 166us/step - loss: 1.1195e-05
mse at 3 1.1122521755169146e-05
day 3
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 166us/step - loss: 7.7813e-06
mse at 2 7.798166734573897e-06
day 2
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 177us/step - loss: 5.5290e-06
mse at 1 5.5678551689197775e-06
day 1
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 166us/